# Example 8.7 / 1: Measure Residuals and Basis Conditioning

Companion notebook for **8.7 Numerical Accuracy** in the AMPL lab Documentation, section "Measure Residuals and Basis Conditioning". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Measure Residuals and Basis Conditioning

An interior-point solve without crossover stops near, not on, the optimal vertex
(How LP Solvers Work (AMPL lab documentation, section 8.9)), which makes its residuals visible. This example solves the
steel model that way and evaluates the violations of `Time` and of the variable bounds from
the full-precision values. In `data.py` and `run.py`, `ampl` is an `AMPL` object that has already
read `model.mod`; `ampl.get_value(...)` evaluates an AMPL expression:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # products
param rate {PROD} > 0;                    # tons produced per hour
param avail >= 0;                         # hours available
param profit {PROD};                      # profit per ton
param market {PROD} >= 0;                 # most tons that can be sold
var Make {p in PROD} >= 0, <= market[p];  # tons produced, up to the market limit
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # total profit
# production hours fit the hours available
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [5]:
# run.py
ampl.solve(solver="highs", highs_options="alg:method=ipm bar:crossover=off pre:solve=off",
           verbose=False)
print(ampl.solve_result, ampl.var["Make"].to_dict())
print("Time violation:", ampl.get_value("max(0, Time.body - avail)"))
print("bound violation:", ampl.get_value("max {p in PROD} max(0, -Make[p], Make[p] - market[p])"))
print("Time slack:", ampl.get_value("Time.slack"))

solved {'bands': 5999.999924648469, 'coils': 4.318594312209025e-05, 'plate': 1600.000006248246}
Time violation: 0
bound violation: 0
Time slack: 2.9235096121738025e-08


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;
param profit := bands 25  coils 30  plate 29;
param market := bands 6000  coils 4000  plate 3500;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
option display_precision 10;
option highs_options 'alg:method=ipm bar:crossover=off pre:solve=off';
solve;
display solve_result, Make;
display max(0, Time.body - avail), Time.slack;

HiGHS 1.14.0:   lp:method = ipm
  bar:crossover = off
  pre:solve = off
solve_result = solved

Make [*] :=
bands  5999.999925
coils     4.318594312e-05
plate  1600.000006
;

max(0,(Time.body) - avail) = 0
Time.slack = 2.923509612e-08



### Another way in amplpy

No constraint is violated, but the plan is not the vertex `6000, 0, 1600` either: `coils` is
about `4e-5` tons and `Time` has a slack of about `3e-8` hours. Both are far below anything
that matters for a steel plan, so rounding to the vertex is safe here; check that the rounded
plan is still feasible in the original model, as it is. The same residual in a constraint
measured in thousands of tons or in a big-M row could matter, which is why units matter
([Scale the Model](#scale-the-model)).

For an LP solution with a basis, some MP drivers report a basis condition number through
`alg:kappa`. A larger condition number means that small perturbations in the basis system can
produce larger changes in its computed solution; it is a warning signal to examine scaling,
residuals, and sensitivity, not a universal pass/fail threshold. With the same `model.mod` and
`data.py`, Gurobi reports it in the solve message (value `1`) and in the suffix `.kappa` of the
objective and the problem (value `2`); `alg:kappa=3` asks for both:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [11]:
# run.py
ampl.solve(solver="gurobi", gurobi_options="alg:kappa=3", verbose=False)
print(ampl.get_value("solve_message"))
print("kappa:", ampl.get_value("Total_Profit.kappa"))

Gurobi 13.0.0: optimal solution; objective 196400
kappa value: 1
1 simplex iteration

kappa: 1
